# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Accessing metadata
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review the available record sets, fields, and their IDs in the dataset.

In [ ]:
# List all available record sets and their field @ids
print('Record sets available in the dataset:')
for record_set in dataset.record_sets:
    rs_json = record_set.to_json()
    print(f"- RecordSet name: {rs_json['name']}, @id: {rs_json['@id']}")
    print("    Fields:")
    for field in record_set.fields:
        f_json = field.to_json()
        print(f"      • {f_json['name']}: {f_json['@id']}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> **Note**: We'll use the primary survey record set, which contains most of the analysis fields, as our example. Replace the `record_sets_of_interest` below if you wish to explore other record sets from the list above.

In [ ]:
# Find the main survey RecordSet: usually contains demographic and response data
record_sets_of_interest = []
for record_set in dataset.record_sets:
    rs_json = record_set.to_json()
    # Heuristic: pick the record set with 'survey' or 'responses' in the name, or the first one if none match
    if 'survey' in rs_json['name'].lower() or 'response' in rs_json['name'].lower():
        record_sets_of_interest.append(rs_json['@id'])
if not record_sets_of_interest and len(dataset.record_sets) > 0:
    # Fallback to first recordset
    record_sets_of_interest.append(dataset.record_sets[0].to_json()['@id'])

dataframes = {}

for record_set_id in record_sets_of_interest:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Fields available in the record set '{record_sets_of_interest[0]}':")
print(dataframes[record_sets_of_interest[0]].columns.to_list())
dataframes[record_sets_of_interest[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping.

We'll use the GAD-7 total score (`gad7_total`) as an example for numeric analysis, and group by village if available.

In [ ]:
df = dataframes[record_sets_of_interest[0]]
# Replace the field names below with the correct @ids for your analysis

# Example field @ids (replace if different in your dataset!)
gad7_total_field = None
village_field = None

# Find the likely GAD-7 total score and village field from columns
for col in df.columns:
    if 'gad7' in col.lower() and 'total' in col.lower():
        gad7_total_field = col
    if 'village' in col.lower():
        village_field = col

if gad7_total_field is None:
    print("Could not find a GAD-7 total field in columns; please check RecordSet overview.")
else:
    # Filtering: GAD-7 scores greater than a threshold (e.g., clinical interest >10)
    threshold = 10
    filtered_df = df[df[gad7_total_field] > threshold]
    print(f"Filtered records with {gad7_total_field} > {threshold}:")
    print(filtered_df.head())

    # Normalizing numeric field
    filtered_df[f"{gad7_total_field}_normalized"] = (
        filtered_df[gad7_total_field] - filtered_df[gad7_total_field].mean()
    ) / filtered_df[gad7_total_field].std()
    print(f"\nNormalized {gad7_total_field} (z-score):")
    print(filtered_df[[gad7_total_field, f"{gad7_total_field}_normalized"]].head())

    # Group by village, if available
    if village_field is not None:
        grouped_df = filtered_df.groupby(village_field)[gad7_total_field].mean().reset_index()
        print(f"\nMean {gad7_total_field} by {village_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_total_field is not None:
    # Distribution of GAD-7 total scores
    plt.figure(figsize=(8,5))
    sns.histplot(df[gad7_total_field].dropna(), bins=15, kde=True)
    plt.title("Distribution of GAD-7 Total Scores")
    plt.xlabel("GAD-7 Total Score")
    plt.ylabel("Count")
    plt.show()

    # If grouping by village, show boxplot
    if village_field is not None:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=village_field, y=gad7_total_field, data=df)
        plt.title(f"{gad7_total_field} by {village_field}")
        plt.ylabel("GAD-7 Total Score")
        plt.xlabel("Village")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library, referencing record sets and fields by their `@id` where applicable. We performed data extraction, filtering, normalization, grouping, and visualization on mental health survey responses from Kilifi County, Kenya.

Further analysis could include more comprehensive feature engineering and multivariate analytics, or cross-record set joins if desired.

> **Tip**: Refer to the printed RecordSet and field `@id`s above to select different fields or record sets for custom exploration.